# RL Test Flow Optimization — Notebook 1 of 3: Setup + Stage-1

**Runtime**: Kaggle notebook  |  **RL training device**: CPU (intentional)  |  **Steps**: 1–5

| Notebook | Contents | Est. Time |
|----------|----------|----------|
| **NB1 (this)** | Install, Data, Env, Baselines, Stage-1 (4 algos × 200K) | ~6-8 h |
| NB2 | Stage-2 (top-2 × 3 seeds × 500K) + Optuna (30 trials × 100K) | ~7-9 h |
| NB3 | Final (1M) + Curriculum + All Plots + Export | ~4-5 h |

## Why 3 Notebooks?
- Kaggle GPU sessions time out after **12 hours**
- Full pipeline = ~20+ hours of compute
- Each notebook saves models + results to `/kaggle/working/` for the next

> **After NB1 completes**: Go to Output tab → Create Dataset → name it `rl-stage1-results`
> Then add that dataset as input to NB2.

## Problem: Semiconductor Test Flow Optimization
A chip must pass through 200 tests (voltage, timing, thermal, functional, analog).
Running all tests is expensive. An RL agent learns the **optimal test ordering** to
maximize defect detection while minimizing cost — industry-critical for AMD, Micron, Infineon.

## Algorithms Compared
| Algorithm | Type | Key Feature |
|-----------|------|-------------|
| A2C | On-policy | Fast convergence, synchronous |
| DQN | Off-policy | Experience replay, stable |
| PPO | On-policy | Clipped surrogate objective |
| MaskablePPO | On-policy | Action masking for invalid tests |

## Device Note
- Kaggle may show a T4 GPU attached, but SB3 with `MlpPolicy` on this tabular environment is CPU-bound
- Forcing CUDA here causes the exact SB3 warning about poor GPU usage and usually makes training slower
- The repo now forces all RL training to `device='cpu'` unless you explicitly override it

> **Production scale note**: This runs 100K chips × 200 tests (Kaggle T4).
> Production: 1M chips × 1000 tests on AMD MI300X / NVIDIA A100.

## Step 1: Environment Setup

In [ ]:
import subprocess, sys, os
subprocess.run(['pip', 'install', '-q',
    'stable-baselines3[extra]', 'sb3-contrib', 'gymnasium',
    'optuna', 'mlflow', 'pyarrow', 'matplotlib', 'seaborn',
    'pandas', 'numpy', 'torch'], check=True)

# Always fresh clone → picks up latest code fixes
!rm -rf rl-test-flow-optimization
!git clone https://github.com/rajendarmuddasani/rl-test-flow-optimization.git
os.chdir('rl-test-flow-optimization')
sys.path.insert(0, '.')

# Verify fix is present (max_steps guard in environment)
import inspect
from src.environment import TestFlowEnv
assert 'max_steps_per_episode' in inspect.signature(TestFlowEnv.__init__).parameters, \
    'ERROR: Old code without infinite-loop fix! Check git clone.'
print('Code version verified: max_steps_per_episode fix present ✓')

import torch
print(f'PyTorch:        {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:            {torch.cuda.get_device_name(0)}')
    print(f'VRAM:           {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('No Kaggle GPU attached. That is acceptable for this SB3 MLP workload.')

import stable_baselines3 as sb3, sb3_contrib, optuna
print(f'SB3:            {sb3.__version__}')
print(f'sb3-contrib:    {sb3_contrib.__version__}')
print(f'Optuna:         {optuna.__version__}')
print('RL train device: cpu (forced intentionally for SB3 MlpPolicy)')
print('\nAll dependencies installed ✓')


## Step 2: Generate Test Results Dataset

100K chips × 200 tests (Kaggle scale). 70% defect rate across 14 defect types.
Production scale: 1M chips × 1000 tests on AMD EPYC cluster.
> **Note**: The RL environment generates chip profiles on-the-fly from the test catalog.
> This dataset is used for statistical validation and defect rate verification.

In [ ]:
from generate_test_data import generate_dataset
import json

summary = generate_dataset(
    output_dir='data',
    n_chips=100_000,
    n_tests=200,
    defect_rate=0.70,
    seed=42,
    chunk_size=25_000,
    fmt='parquet',
)
print('Dataset Summary:')
print(json.dumps(summary, indent=2))


## Step 3: Environment Validation

Verify the Gymnasium environment scales correctly at 10, 50, and 200 tests.
**Vectorised hot paths**: `_obs()` and `action_masks()` use numpy ops (no Python loops) for 20-50x speedup.

In [ ]:
import numpy as np, time
from src.environment import TestFlowEnv, DEFECT_CATEGORIES, CATEGORY_GROUPS

print('Environment Validation')
print('=' * 60)

for n in [10, 50, 200]:
    env = TestFlowEnv(n_tests=n, cost_budget=50.0, time_budget=120.0)
    obs, _ = env.reset(seed=42)
    masks = env.action_masks()
    print(f'n_tests={n:4d}: obs={obs.shape[0]}d  actions={env.action_space.n}  valid={int(masks.sum())}')

# Benchmark environment speed
print('\n=== Environment Speed Benchmark ===')
env200 = TestFlowEnv(n_tests=200, cost_budget=50.0, time_budget=120.0)
obs, _ = env200.reset(seed=42)
N_BENCH = 10_000
t0 = time.perf_counter()
for _ in range(N_BENCH):
    mask = env200.action_masks()
    valid = np.where(mask)[0]
    action = int(np.random.choice(valid))
    obs, r, done, trunc, info = env200.step(action)
    if done or trunc:
        obs, _ = env200.reset()
elapsed = time.perf_counter() - t0
print(f'{N_BENCH:,} steps in {elapsed:.2f}s  →  {N_BENCH/elapsed:.0f} steps/sec')
print(f'Estimated time for 200K training steps: {200_000/(N_BENCH/elapsed)/60:.1f} min')
print('\nEnvironment checks passed ✓')


## Step 4: Baseline Evaluation

Three heuristic policies evaluated over 500 episodes. RL agents must beat these to be useful.

| Baseline | Strategy |
|----------|----------|
| Random | Uniformly random from valid tests |
| Greedy Coverage | Always highest-coverage test |
| Cost Efficient | Best coverage/cost ratio up to 40% budget |

In [ ]:
import pandas as pd
from src.agent import BASELINES, evaluate_policy

env = TestFlowEnv(n_tests=200, cost_budget=50.0, time_budget=120.0)

print('Baseline Evaluation (500 episodes each)')
print('=' * 70)
print(f'{"Policy":20s} {"Reward":>10s} {"Accuracy":>10s} {"Cost":>10s} {"Tests":>10s}')
print('-' * 70)

baseline_results = {}
for name, policy_fn in BASELINES.items():
    metrics = evaluate_policy(env, policy_fn, n_episodes=500)
    baseline_results[name] = metrics
    print(f'{name:20s} {metrics["mean_reward"]:>+10.2f} {metrics["accuracy"]:>10.3f} '
          f'{metrics["mean_cost"]:>10.2f} {metrics["mean_tests"]:>10.1f}')

print('=' * 70)
baseline_df = pd.DataFrame(baseline_results).T
best_bl = baseline_df['mean_reward'].max()
print(f'Best baseline: {baseline_df["mean_reward"].idxmax()} (reward={best_bl:+.2f})')
print(f'\nRL must beat: {best_bl:+.2f} reward')


## Step 5: Stage-1 — Train All 4 Algorithms (200K steps each)

**Execution order**: A2C → DQN → PPO → MaskablePPO (fastest to slowest).
Progress prints every 50K steps with elapsed time and ETA.
Training is forced to CPU on purpose to avoid SB3's slow-CUDA warning for MLP policies.

| Algorithm | Key Setting | Expected Time |
|-----------|------------|---------------|
| A2C | n_steps=5, fast updates | ~7-15 min |
| DQN | buffer=50K, replay | ~15-25 min |
| PPO | n_steps=2048, clipped | ~15-25 min |
| MaskablePPO | action masking | ~20-30 min |

> Best-model checkpoints saved to `/kaggle/working/rl_stage1/` every 25K steps via EvalCallback.
> Environment has max_steps guard to prevent infinite loops during evaluation.

In [ ]:
import time as _time, json, sys
import pandas as pd
from pathlib import Path
from src.agent import ALGO_REGISTRY, evaluate_trained_model

def flush_print(*args, **kwargs):
    print(*args, **kwargs)
    sys.stdout.flush()

STAGE1_STEPS = 200_000
SAVE_DIR = Path('/kaggle/working/rl_stage1')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# ── Quick benchmark: estimate actual training speed ──
flush_print('Benchmarking environment speed (2K steps)...')
bench_env = TestFlowEnv(n_tests=200, cost_budget=50.0, time_budget=120.0)
from stable_baselines3 import A2C
from stable_baselines3.common.monitor import Monitor
bench_model = A2C('MlpPolicy', Monitor(bench_env), verbose=0, device='cpu')
t_bench = _time.time()
bench_model.learn(total_timesteps=2000)
bench_fps = 2000 / (_time.time() - t_bench)
est_per_algo = STAGE1_STEPS / bench_fps / 60
est_total = est_per_algo * 4
flush_print(f'Measured speed: {bench_fps:.0f} steps/sec')
flush_print(f'Estimated time: ~{est_per_algo:.0f} min/algo x 4 = ~{est_total:.0f} min total')
flush_print(f'(+ ~2 min eval overhead per algo)')
del bench_model, bench_env

stage1_results = {}
algo_list = list(ALGO_REGISTRY.keys())

flush_print(f'\nStage-1: Training {len(algo_list)} algorithms x {STAGE1_STEPS:,} steps')
flush_print('=' * 70)
overall_t0 = _time.time()

for idx, algo_name in enumerate(algo_list):
    train_fn = ALGO_REGISTRY[algo_name]
    flush_print(f'\n[{idx+1}/{len(algo_list)}] >>> {algo_name.upper()}')
    flush_print(f'  Starting at {_time.strftime("%H:%M:%S")}  |  '
               f'Est. ~{est_per_algo:.0f} min')
    env = TestFlowEnv(n_tests=200, cost_budget=50.0, time_budget=120.0)
    t0 = _time.time()
    model = train_fn(
        env,
        total_timesteps=STAGE1_STEPS,
        output_dir=str(SAVE_DIR / algo_name),
    )
    train_time = _time.time() - t0
    flush_print(f'  Training done in {train_time/60:.1f}m. Evaluating 100 episodes...')
    metrics = evaluate_trained_model(env, model, n_episodes=100)
    metrics['train_time_sec'] = round(train_time, 1)
    stage1_results[algo_name] = metrics
    elapsed_total = (_time.time() - overall_t0) / 60
    flush_print(f'  DONE: reward={metrics["mean_reward"]:+.2f} | '
               f'acc={metrics["accuracy"]:.3f} | '
               f'cost={metrics["mean_cost"]:.2f} | '
               f'train={train_time/60:.1f}m | '
               f'total_elapsed={elapsed_total:.1f}m')

# Save results
stage1_df = pd.DataFrame(stage1_results).T
ranked = stage1_df.sort_values('mean_reward', ascending=False)
top2_algos = list(ranked.index[:2])
total_elapsed = (_time.time() - overall_t0) / 60

flush_print(f'\n{"="*70}')
flush_print(f'STAGE-1 COMPLETE  |  Total wall time: {total_elapsed:.1f} min')
flush_print(stage1_df.to_string())
flush_print(f'\nTop-2 for Stage-2: {top2_algos}')

for algo, row in stage1_df.iterrows():
    imp = row['mean_reward'] - best_bl
    pct = (imp / abs(best_bl)) * 100 if best_bl != 0 else 0
    flush_print(f'  {algo} vs best baseline: {imp:+.2f} ({pct:+.1f}%)')


## Save Outputs for NB2

All models already saved to `/kaggle/working/rl_stage1/` by EvalCallback.
Now save the metrics JSON and create the summary.

> **After this cell**: Output tab → Create Dataset → name it **`rl-stage1-results`**
> Add it as input to NB2.

In [ ]:
import json, shutil
from pathlib import Path

save_data = {
    'baselines': baseline_results,
    'stage1':    stage1_results,
    'top2_algos': top2_algos,
    'best_base_reward': float(best_bl),
}

with open(SAVE_DIR / 'stage1_results.json', 'w') as f:
    json.dump(save_data, f, indent=2)

print('=== NB1 ARTIFACTS ===')
for f in sorted(SAVE_DIR.rglob('*')):
    if f.is_file():
        print(f'  {str(f.relative_to(SAVE_DIR)):60s} {f.stat().st_size/1e3:6.0f} KB')

total = sum(f.stat().st_size for f in SAVE_DIR.rglob('*') if f.is_file())
print(f'\nTotal: {total/1e6:.1f} MB')
print('\nNB1 complete!')
print('Next: Output tab → Create Dataset → name it rl-stage1-results')
print('Then open NB2 and add rl-stage1-results as input dataset')
